In [ ]:
import pandas as pd
import sqlite3
import os

raw_path = '../data/raw/'
files = os.listdir(raw_path)
csv_files = [f for f in files if f.endswith('.csv')]
df = pd.read_csv(raw_path + csv_files[0])

conn = sqlite3.connect('../data/processed/bank.db')
df.to_sql('clients', conn, if_exists='replace', index=False)


Данные загружены в SQLite
Таблица clients: 10000 записей


In [3]:
query = """
SELECT 
    COUNT(*) AS total_clients,
    SUM(churn) AS total_churned,
    ROUND(AVG(churn) * 100, 2) AS churn_rate,
    ROUND(AVG(age), 2) AS avg_age,
    ROUND(AVG(balance), 2) AS avg_balance
FROM clients
"""

pd.read_sql_query(query, conn)

,total_clients,total_churned,churn_rate,avg_age,avg_balance
0,10000,2037,20.37,38.92,76485.89


Запрос показывает общее количество клиентов, количество ушедших, долю оттока, средний возраст и средний баланс.

In [4]:
query = """
SELECT 
    country,
    COUNT(*) AS total,
    SUM(churn) AS churned,
    ROUND(AVG(churn) * 100, 2) AS churn_rate
FROM clients
GROUP BY country
ORDER BY churn_rate DESC
"""

pd.read_sql_query(query, conn)

,country,total,churned,churn_rate
0,Germany,2509,814,32.44
1,Spain,2477,413,16.67
2,France,5014,810,16.15


Запрос показывает долю оттока в каждой стране. Германия лидирует по оттоку.

In [5]:
query = """
SELECT 
    products_number,
    COUNT(*) AS total,
    SUM(churn) AS churned,
    ROUND(AVG(churn) * 100, 2) AS churn_rate
FROM clients
GROUP BY products_number
ORDER BY products_number
"""

pd.read_sql_query(query, conn)

,products_number,total,churned,churn_rate
0,1,5084,1409,27.71
1,2,4590,348,7.58
2,3,266,220,82.71
3,4,60,60,100.00


Запрос подтверждает результаты A/B-теста. Клиенты с одним продуктом уходят чаще.

In [6]:
query = """
SELECT 
    CASE 
        WHEN age < 30 THEN '18-29'
        WHEN age < 40 THEN '30-39'
        WHEN age < 50 THEN '40-49'
        WHEN age < 60 THEN '50-59'
        ELSE '60+'
    END AS age_group,
    COUNT(*) AS total,
    SUM(churn) AS churned,
    ROUND(AVG(churn) * 100, 2) AS churn_rate
FROM clients
GROUP BY age_group
ORDER BY age_group
"""

pd.read_sql_query(query, conn)

,age_group,total,churned,churn_rate
0,18-29,1641,124,7.56
1,30-39,4346,473,10.88
2,40-49,2618,806,30.79
3,50-59,869,487,56.04
4,60+,526,147,27.95


Запрос показывает, что клиенты 40-59 лет уходят чаще остальных.

In [7]:
query = """
SELECT 
    customer_id,
    age,
    balance,
    products_number,
    churn
FROM clients
ORDER BY balance DESC
LIMIT 10
"""

pd.read_sql_query(query, conn)

,customer_id,age,balance,products_number,churn
0,15757408,38,250898.09,3,1
1,15715622,57,238387.56,1,1
2,15714241,42,222267.63,1,1
3,15571958,40,221532.80,1,0
4,15586674,58,216109.88,1,1
5,15599131,26,214346.96,2,0
6,15594408,48,213146.20,1,1
7,15769818,37,212778.20,1,0
8,15620268,43,212696.32,1,0
9,15780212,37,212692.97,1,0


Запрос показал десять клиентов с наибольшим балансом. Шесть из десяти ушли, это 60% топ-клиентов. Возраст ушедших от 38 до 58 лет, что попадает в группу риска 40-59 лет. У большинства ушедших только один продукт.Банк теряет самых богатых клиентов с балансом от 212 до 250 тысяч. Это серьёзная потеря для бизнеса.

In [8]:
query = """
SELECT 
    customer_id,
    country,
    balance,
    RANK() OVER (PARTITION BY country ORDER BY balance DESC) AS rank_in_country
FROM clients
LIMIT 20
"""

pd.read_sql_query(query, conn)

,customer_id,country,balance,rank_in_country
0,15715622,France,238387.56,1
1,15769818,France,212778.20,2
2,15780212,France,212692.97,3
3,15690589,France,212314.03,4
4,15671256,France,211774.31,5
5,15736420,France,210433.08,6
6,15709920,France,208165.53,7
7,15627971,France,206663.75,8
8,15784180,France,206329.65,9
9,15664498,France,205962.00,10


Использована оконная функция RANK для ранжирования клиентов по балансу внутри каждой страны. Параметр PARTITION BY country делит данные на группы, ORDER BY balance DESC сортирует по убыванию баланса. Запрос показывает, кто является самым богатым клиентом в каждой стране.

In [9]:
conn.close()